# 원본 데이터 전처리

`okm_augumented_2021.csv`(2021년 1월~9월, 1시간 단위 공장 전력·생산·기상 데이터)를 점검하고 정제해
이후 모든 분석·모델이 사용하는 **`okm_augumented_2021_processed.xlsx`** 를 만든다.

| 단계 | 내용 |
|---|---|
| 1 | 컬럼 요약 (자료형 · 결측 · 범위) |
| 2 | 날짜·시간 구조 점검 및 잘못된 `시간` 값 복원 |
| 3 | 15분 단위 전력값으로 파생 변수 생성 |
| 4 | 결측치 처리 (풍속 · 강수량 · 공장인원) |
| 5 | 공휴일을 요일 코드 `8`로 변경 |
| 6 | 최종 점검 및 저장 |

- **입력**: `okm_augumented_2021.csv`
- **출력**: `okm_augumented_2021_processed.xlsx`

## 0. 데이터 불러오기

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SRC = "okm_augumented_2021.csv"
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 100)

raw = pd.read_csv(SRC, encoding="utf-8-sig")
df = raw.copy()
print(raw.shape)
raw.head()

(6168, 18)


,날짜,시간,15분,30분,45분,60분,평균,생산량,기온,풍속,습도,강수량,전기요금(계절),day,d,m,공장인원,인건비
0,20210101,0,62,61,61,61,61,0,-3.2,2.4,71,0.0,109.8,5,1,1,0.0,1.5
1,20210101,1,96,93,116,113,105,0,-4.5,1.5,77,0.0,109.8,5,1,1,0.0,1.5
2,20210101,2,106,96,106,107,104,0,-3.9,2.6,58,0.0,109.8,5,1,1,0.0,1.5
3,20210101,3,92,110,110,109,105,0,-4.1,2.6,56,0.0,109.8,5,1,1,0.0,1.5
4,20210101,4,108,105,106,108,107,0,-4.6,2.6,60,0.0,109.8,5,1,1,0.0,1.5


## 1. 컬럼 요약 (자료형 · 결측 · 범위)

In [2]:
summary = pd.DataFrame({
    "dtype" : raw.dtypes.astype(str),
    "결측수": raw.isna().sum(),
    "고유값 수": raw.nunique(),
    "최소" : raw.min(numeric_only=True),
    "최대" : raw.max(numeric_only=True),
})
summary

,dtype,결측수,고유값 수,최소,최대
날짜,int64,0,257,20210101.0,2.021091e+07
시간,int64,0,60,0.0,1.880000e+02
15분,int64,0,166,0.0,2.070000e+02
30분,int64,0,185,0.0,2.220000e+02
45분,int64,0,187,0.0,2.180000e+02
60분,int64,0,183,0.0,2.140000e+02
평균,int64,0,176,0.0,2.080000e+02
생산량,int64,0,1637,0.0,9.830000e+03
기온,float64,0,439,-12.0,3.340000e+01
풍속,float64,3,70,0.0,7.600000e+00


## 2. 날짜 · 시간 구조 점검

### 2-1. `시간` 값 범위 확인
`시간`은 0~23이어야 하지만 일부 날짜에 범위를 벗어난 값이 있다.

In [3]:
print("시간 최소/최대:", df["시간"].min(), df["시간"].max())
bad_mask = ~df["시간"].between(0,23)
print("범위 밖 행 수:", bad_mask.sum())
print("해당 날짜:\n", df.loc[bad_mask, "날짜"].value_counts())

시간 최소/최대: 0 188
범위 밖 행 수: 48
해당 날짜:
 날짜
20210713    24
20210715    24
Name: count, dtype: int64


### 2-2. 날짜별 행 수 · (날짜, 시간) 중복 확인

In [4]:
print("날짜별 행 수 분포:\n", df.groupby("날짜").size().value_counts())
dup = df[df.duplicated(["날짜", "시간"], keep=False)]
print("\n(날짜, 시간) 중복 행 수:", len(dup))
dup[["날짜", "시간", "평균"]]

날짜별 행 수 분포:
 24    257
Name: count, dtype: int64

(날짜, 시간) 중복 행 수: 10


,날짜,시간,평균
4649,20210713,159,143
4650,20210713,159,152
4653,20210713,172,174
4654,20210713,172,178
4685,20210715,110,88
4686,20210715,110,103
4688,20210715,115,134
4689,20210715,115,113
4690,20210715,120,108
4691,20210715,120,134


### 2-3. 문제 날짜가 온전한 하루치(24행)인지 확인

In [5]:
bad_dates = df.loc[bad_mask, "날짜"].unique()
rows_per_bad_date = df[df["날짜"].isin(bad_dates)].groupby("날짜").size()
print(rows_per_bad_date)
print("모두 24행인가:", (rows_per_bad_date == 24).all())

날짜
20210713    24
20210715    24
dtype: int64
모두 24행인가: True


### 2-4. `시간` 복원
문제 날짜는 모두 24행이고 행 순서가 시간 순서와 같으므로, 행 순서대로 0~23을 다시 부여한다.
복원한 행은 `시간복원 = 1`로 표시한다.

In [6]:
bad_dates = df.loc[bad_mask, "날짜"].unique()
df["시간복원"] = 0
for d in bad_dates:
    m = df["날짜"] == d
    assert m.sum() == 24 and df.loc[m, "시간"].is_monotonic_increasing
    df.loc[m, "시간"] = np.arange(24)
    df.loc[m, "시간복원"] = 1
print("복원 행 수:", df["시간복원"].sum())
print("시간 최소/최대:", df["시간"].min(), df["시간"].max())
df[df["날짜"] == 20210713][["날짜", "시간", "평균", "시간복원"]].head(6)

복원 행 수: 48
시간 최소/최대: 0 23


,날짜,시간,평균,시간복원
4632,20210713,0,69,1
4633,20210713,1,96,1
4634,20210713,2,112,1
4635,20210713,3,110,1
4636,20210713,4,106,1
4637,20210713,5,143,1


### 2-5. `datetime` 생성 및 1시간 간격 연속성 검증

In [7]:
df["datetime"] = pd.to_datetime(df["날짜"].astype(str)) + pd.to_timedelta(df["시간"], unit="h")
df = df.sort_values("datetime").reset_index(drop=True)

print("(datetime) 중복:", df["datetime"].duplicated().sum())
gaps = df["datetime"].diff().dropna()
print("1시간 간격이 아닌 구간:", (gaps != pd.Timedelta(hours=1)).sum())

full_days = pd.date_range(df["datetime"].dt.normalize().min(), df["datetime"].dt.normalize().max())
print("누락 날짜:", list(full_days.difference(df["datetime"].dt.normalize().unique())))
print("기간:", df["datetime"].min(), "~", df["datetime"].max())

(datetime) 중복: 0
1시간 간격이 아닌 구간: 0
누락 날짜: []
기간: 2021-01-01 00:00:00 ~ 2021-09-14 23:00:00


### 2-6. `day`(요일) · `d`(일) · `m`(월) 컬럼이 날짜와 일치하는지 확인

In [8]:
dow = df["datetime"].dt.dayofweek+1   # 월=1 ~ 일=7
print("day 불일치:", (df["day"] != dow).sum())
print("d   불일치:", (df["d"] != df["datetime"].dt.day).sum())
print("m   불일치:", (df["m"] != df["datetime"].dt.month).sum())
df.groupby("m").size()

day 불일치: 0
d   불일치: 0
m   불일치: 0


m
1    744
2    672
3    744
4    720
5    744
6    720
7    744
8    744
9    336
dtype: int64

### 2-7. 복원 결과 재검증

In [9]:
print("시간 최소/최대:", df["시간"].min(), df["시간"].max())
bad_mask = ~df["시간"].between(0,23)
print("범위 밖 행 수:", bad_mask.sum())
print("해당 날짜:\n", df.loc[bad_mask, "날짜"].value_counts())
print("날짜별 행 수 분포:\n", df.groupby("날짜").size().value_counts())
dup = df[df.duplicated(["날짜", "시간"], keep=False)]
print("\n(날짜, 시간) 중복 행 수:", len(dup))
dup[["날짜", "시간", "평균"]]

시간 최소/최대: 0 23
범위 밖 행 수: 0
해당 날짜:
 Series([], Name: count, dtype: int64)
날짜별 행 수 분포:
 24    257
Name: count, dtype: int64

(날짜, 시간) 중복 행 수: 0


,날짜,시간,평균


## 3. 전력 파생 변수 생성
15분 단위 전력값(`15분`~`60분`)으로 시간 단위 파생 변수를 만든다.

| 변수 | 정의 |
|---|---|
| `전력 구간합계` | 15·30·45·60분 값의 합 |
| `전력 최대값` / `전력 최소값` | 4개 값 중 최대 / 최소 |
| `전력 변동폭` | 최대값 − 최소값 |

In [10]:
power_cols = ["15분", "30분", "45분", "60분", "평균"]
df_power = df.copy()

for col in power_cols:
    df_power[col] = pd.to_numeric(df_power[col], errors="coerce")

df_power["전력 구간합계"] = df_power["15분"] + df_power["30분"] + df_power["45분"] + df_power["60분"]
df_power["전력 최대값"] = df_power[["15분", "30분", "45분", "60분"]].max(axis=1)
df_power["전력 최소값"] = df_power[["15분", "30분", "45분", "60분"]].min(axis=1)
df_power["전력 변동폭"] = df_power["전력 최대값"] - df_power["전력 최소값"]

print("---전력 데이터 전처리 완료. 생성된 컬럼 확인---")
print(df_power[["15분", "30분", "45분", "60분", "평균", "전력 구간합계", "전력 변동폭"]].head())

df = df_power.copy()

---전력 데이터 전처리 완료. 생성된 컬럼 확인---
   15분  30분  45분  60분   평균  전력 구간합계  전력 변동폭
0   62   61   61   61   61      245       1
1   96   93  116  113  105      418      23
2  106   96  106  107  104      415      11
3   92  110  110  109  105      421      18
4  108  105  106  108  107      427       3


## 4. 결측치 처리

### 4-1. 풍속 — 선형 보간

In [11]:
print("보간 전 풍속 결측 수:", df["풍속"].isna().sum())

df["풍속"] = df["풍속"].interpolate(method="linear")
df["풍속"] = df["풍속"].ffill().bfill()

print("보간 후 풍속 결측 수:", df["풍속"].isna().sum())

보간 전 풍속 결측 수: 3
보간 후 풍속 결측 수: 0


### 4-2. 강수량 — 선형 보간

In [12]:
print("보간 전 강수량 결측 수:", df["강수량"].isna().sum())
df["강수량"] = df["강수량"].interpolate(method="linear")
df["강수량"] = df["강수량"].ffill().bfill()
print("보간 후 강수량 결측 수:", df["강수량"].isna().sum())

보간 전 강수량 결측 수: 1
보간 후 강수량 결측 수: 0


### 4-3. 공장인원 결측 구간 확인
결측은 2021-08-28 18시 ~ 08-29 10시(주말)에 몰려 있고, 이 구간은 생산량·전력이 모두 0이다.

In [13]:
missing_employee = df[df["공장인원"].isna()]
print(f"공장인원 결측치 총 개수: {len(missing_employee)}개")

if len(missing_employee) > 0:
    # 결측 구간 앞뒤 5행까지 함께 출력
    first_idx = missing_employee.index[0]
    last_idx = missing_employee.index[-1]
    start_range = max(0, first_idx - 5)
    end_range = min(len(df) - 1, last_idx + 5)
    print(f"\n--- 공장인원 결측치 및 전후 구간 (인덱스 {start_range} ~ {end_range}) ---")
    display_cols = ["datetime", "날짜", "시간", "공장인원", "생산량", "인건비", "day"]
    display(df.loc[start_range:end_range, display_cols])

공장인원 결측치 총 개수: 17개

--- 공장인원 결측치 및 전후 구간 (인덱스 5749 ~ 5775) ---


,datetime,날짜,시간,공장인원,생산량,인건비,day
5749,2021-08-28 13:00:00,20210828,13,0.0,0,1.0,6
5750,2021-08-28 14:00:00,20210828,14,0.0,0,1.0,6
5751,2021-08-28 15:00:00,20210828,15,0.0,0,1.0,6
5752,2021-08-28 16:00:00,20210828,16,0.0,0,1.0,6
5753,2021-08-28 17:00:00,20210828,17,0.0,0,1.0,6
5754,2021-08-28 18:00:00,20210828,18,NaN,0,1.5,6
5755,2021-08-28 19:00:00,20210828,19,NaN,0,1.5,6
5756,2021-08-28 20:00:00,20210828,20,NaN,0,1.5,6
5757,2021-08-28 21:00:00,20210828,21,NaN,0,1.5,6
5758,2021-08-28 22:00:00,20210828,22,NaN,0,1.5,6


### 4-4. 공장인원 결측 대체 — RandomForest 회귀
결측 구간과 같은 조건인 **주말 데이터**로 모델을 학습해 결측값을 예측한다.

In [14]:
from sklearn.ensemble import RandomForestRegressor

print("대체 전 공장인원 결측 수:", df["공장인원"].isna().sum())

features = ["생산량", "평균", "시간", "day", "습도", "기온", "강수량"]

# 학습 데이터: 공장인원 값이 있는 주말(토·일) 시간
train_df = df[df["day"].isin([6, 7]) & df["공장인원"].notna()]
X_train = train_df[features]
y_train = train_df["공장인원"]

# 예측 대상: 공장인원 결측 시간
test_df = df[df["공장인원"].isna()]
X_test = test_df[features]

# RandomForest 학습
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# 결측 구간 채우기
predicted_values = model.predict(X_test)
df.loc[df["공장인원"].isna(), "공장인원"] = predicted_values
print("대체 후 공장인원 결측 수:", df["공장인원"].isna().sum())

print(df.loc[test_df.index, ["datetime", "생산량", "평균", "공장인원"]].head(10))

대체 전 공장인원 결측 수: 17
대체 후 공장인원 결측 수: 0
                datetime  생산량  평균  공장인원
5754 2021-08-28 18:00:00    0   0   0.0
5755 2021-08-28 19:00:00    0   0   0.0
5756 2021-08-28 20:00:00    0   0   0.0
5757 2021-08-28 21:00:00    0   0   0.0
5758 2021-08-28 22:00:00    0   0   0.0
5759 2021-08-28 23:00:00    0   0   0.0
5760 2021-08-29 00:00:00    0   0   0.0
5761 2021-08-29 01:00:00    0   0   0.0
5762 2021-08-29 02:00:00    0   0   0.0
5763 2021-08-29 03:00:00    0   0   0.0


## 5. 공휴일 처리
공휴일(대체공휴일 포함)의 요일 코드 `day`를 **8**로 바꾼다. (1~7 = 월~일, 8 = 공휴일)

In [20]:
# 2021년 공휴일 (대체공휴일 포함)
holiday_list = [
    20210101,  # 신정
    20210211, 20210212, 20210213,  # 설날 연휴
    20210301,  # 3·1절
    20210505,  # 어린이날
    20210519,  # 부처님오신날
    20210815, 20210816,  # 광복절 및 대체공휴일
    20210920, 20210921, 20210922,  # 추석 연휴
]

holiday_list = sorted(list(set(holiday_list)))

df["날짜"] = df["날짜"].astype(int)
mask = df["날짜"].isin(holiday_list)
changed_count = mask.sum()
df.loc[mask, "day"] = 8

print(f"공휴일 및 대체공휴일로 인해 요일이 공휴일(8)로 변경된 데이터 행 수: {changed_count}개")

공휴일 및 대체공휴일로 인해 요일이 공휴일(8)로 변경된 데이터 행 수: 216개


## 6. 최종 점검

In [21]:
summary = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "결측수": df.isna().sum(),
    "고유값 수": df.nunique(),
    "최소": df.min(numeric_only=True),
    "최대": df.max(numeric_only=True),
})

print("--- 최종 데이터 요약 및 오류 확인 ---")
print(summary)
print(f"\n전체 데이터 결측치 총합: {df.isna().sum().sum()}개")

--- 최종 데이터 요약 및 오류 확인 ---
                   dtype  결측수  고유값 수          최소            최대
15분                int64    0    166         0.0  2.070000e+02
30분                int64    0    185         0.0  2.220000e+02
45분                int64    0    187         0.0  2.180000e+02
60분                int64    0    183         0.0  2.140000e+02
d                  int64    0     31         1.0  3.100000e+01
datetime  datetime64[ns]    0   6168         NaN           NaN
day                int64    0      8         1.0  8.000000e+00
m                  int64    0      9         1.0  9.000000e+00
강수량              float64    0    245         0.0  1.224000e+02
공장인원             float64    0   3432         0.0  4.838636e+01
기온               float64    0    439       -12.0  3.340000e+01
날짜                 int64    0    257  20210101.0  2.021091e+07
생산량                int64    0   1637         0.0  9.830000e+03
습도                 int64    0     90         8.0  9.800000e+01
시간                 int64    0

## 7. 저장

In [22]:
output_filename = "okm_augumented_2021_processed.xlsx"
df.to_excel(output_filename, index=False, engine='openpyxl')

In [23]:
# 엔진을 지정하지 않고 저장 (위 셀과 결과 동일)
output_filename = "okm_augumented_2021_processed.xlsx"
df.to_excel(output_filename, index=False)

### 저장 파일 확인

In [24]:
NSRC = "okm_augumented_2021_processed.xlsx"
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 100)

ndf = pd.read_excel(NSRC)
ndf.head()

,날짜,시간,15분,30분,45분,60분,평균,생산량,기온,풍속,습도,강수량,전기요금(계절),day,d,m,공장인원,인건비,시간복원,datetime,전력 구간합계,전력 최대값,전력 최소값,전력 변동폭
0,20210101,0,62,61,61,61,61,0,-3.2,2.4,71,0.0,109.8,8,1,1,0.0,1.5,0,2021-01-01 00:00:00,245,62,61,1
1,20210101,1,96,93,116,113,105,0,-4.5,1.5,77,0.0,109.8,8,1,1,0.0,1.5,0,2021-01-01 01:00:00,418,116,93,23
2,20210101,2,106,96,106,107,104,0,-3.9,2.6,58,0.0,109.8,8,1,1,0.0,1.5,0,2021-01-01 02:00:00,415,107,96,11
3,20210101,3,92,110,110,109,105,0,-4.1,2.6,56,0.0,109.8,8,1,1,0.0,1.5,0,2021-01-01 03:00:00,421,110,92,18
4,20210101,4,108,105,106,108,107,0,-4.6,2.6,60,0.0,109.8,8,1,1,0.0,1.5,0,2021-01-01 04:00:00,427,108,105,3
